In [1]:
# [셀1] import + 상수
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

ID_COL, TARGET_COL, C20_COL = "C64", "C65", "C20"
FMT = "%Y-%m-%d %H:%M:%S.%f"
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33", "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]
SEEDS = [1, 2, 3]
SIGMA_C65 = 261.7
TRAIN_CUT = pd.Timestamp("2019-01-14")   # 학습기 상한 (후기 홀드아웃 미접근)
XGB_DEVICE = os.environ.get("XGB_DEVICE", "cpu")

def _find(name):
    for d in [".", "data", "colab_GA", "..", os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"), os.path.join("..", "modeling_v13", "colab_GA"),
              os.path.join("..", "문제1(하)")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz"); META = _find("core10_meta_wf.csv")
DIET = _find("feature_diet_selected.json"); SEL = _find("select_result_Conservative_GA.json")
XGBJSON = _find("tuned_params_v13_xgb.json"); LEANJSON = _find("lean_timestable_set.json"); TRAIN = _find("train_data.csv")
print("xgboost", xgb.__version__, "| device", XGB_DEVICE)


xgboost 3.3.0 | device cpu


In [2]:
# [셀2] 헬퍼
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c)
    return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

def r2_honest(rmse):
    return round(1 - (rmse / SIGMA_C65) ** 2, 4)

def stable_group_kfold(groups, n_splits=5):
    groups = np.asarray(groups)
    _, inv, cnt = np.unique(groups, return_inverse=True, return_counts=True)
    order = np.argsort(cnt, kind="stable")[::-1]
    fs = np.zeros(n_splits, dtype=np.int64); g2f = np.zeros(len(cnt), dtype=np.int64)
    for gi in order:
        f = int(np.argmin(fs)); fs[f] += cnt[gi]; g2f[gi] = f
    return g2f[inv]

def stable_splits(groups, n_splits=5):
    fv = stable_group_kfold(groups, n_splits)
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]

def lot_seed_splits(groups, seed, n_splits=5):
    groups = np.asarray(groups); lots = np.unique(groups); lf = np.empty(len(lots), dtype=np.int64)
    for f, (_, te) in enumerate(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(lots)):
        lf[te] = f
    l2f = {l: int(fo) for l, fo in zip(lots, lf)}
    fv = np.array([l2f[g] for g in groups])
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]

def make_xgb(params, rounds):
    p = dict(params)
    p.update(objective="reg:squarederror", tree_method="hist", device=XGB_DEVICE,
             random_state=42, n_estimators=int(rounds))
    return xgb.XGBRegressor(**p)

def oof_xgb(cols, rounds, splits, df, y):
    oof = np.zeros(len(y)); Xc = df[cols]
    for tr, va in splits:
        m = make_xgb(XGB_PARAMS, rounds); m.fit(Xc.iloc[tr], y[tr]); oof[va] = m.predict(Xc.iloc[va])
    return _rmse(y, oof)


In [3]:
# [셀3] 로드 + 3개 셋 구성 + 웨이퍼 시간 + asserts
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
tt = pd.read_csv(TRAIN, usecols=[ID_COL, "C40"]); tt[ID_COL] = tt[ID_COL].astype(str)
tt["_ts"] = pd.to_datetime(tt["C40"], format=FMT)
df = df.merge(tt.groupby(ID_COL)["_ts"].min().reset_index().rename(columns={"_ts": "wf_ts"}), on=ID_COL, how="inner").reset_index(drop=True)
y = df[TARGET_COL].to_numpy(float); groups = df[C20_COL].to_numpy()

champions = list(json.load(open(DIET, encoding="utf-8"))["champions"]["Conservative"].values())
fixed = [f for f in dict.fromkeys(CORE10 + champions) if f in df.columns]
selp = json.load(open(SEL, encoding="utf-8"))["selected_features"]
SET_CHAMP = list(dict.fromkeys(fixed + selp))                       # 99
SET_LEAN = json.load(open(LEANJSON, encoding="utf-8"))["lean_features"]   # 85
SET_CORE = [f for f in CORE10 if f in df.columns]                   # 10
XGB_PARAMS = json.load(open(XGBJSON, encoding="utf-8"))["params"]
XGB_ROUNDS = int(json.load(open(XGBJSON, encoding="utf-8"))["n_estimators"])

for nm, s in [("champ99", SET_CHAMP), ("lean", SET_LEAN)]:
    ok, have = floor_ok(s)
    assert all(f in df.columns for f in s), f"{nm} 누락 피처"
    assert C20_COL not in s and ID_COL not in s and "fold_kf5" not in s, f"{nm} 누수!"
    assert ok, f"{nm} floor 위반 {have}"
    print(f"  {nm}: n={len(s)} floor={have}")
print(f"  core10: n={len(SET_CORE)}")

STABLE = stable_splits(groups); SEED_SP = {s: lot_seed_splits(groups, s) for s in SEEDS}
# 학습기 내부 early/late (홀드아웃 무소비)
tr_mask = (df["wf_ts"] <= TRAIN_CUT).to_numpy()
tr_df = df[tr_mask]; med = tr_df["wf_ts"].median()
E = np.where(tr_mask & (df["wf_ts"] <= med).to_numpy())[0]
L = np.where(tr_mask & (df["wf_ts"] > med).to_numpy())[0]
print(f"  df {df.shape} | 학습기 n={int(tr_mask.sum())} | early n={len(E)} late n={len(L)} (≤{TRAIN_CUT.date()})")


  champ99: n=99 floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
  lean: n=85 floor={'C17': 2, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 1}
  core10: n=10
  df (11939, 668) | 학습기 n=6922 | early n=3461 late n=3461 (≤2019-01-14)


In [4]:
# [셀4] lot-CV (같은 시기 새 Lot): stable GKF + 다중시드
print("=== lot-CV (stable GKF + seed1/2/3) — 챔피언 파라미터 공통 ===")
lotcv = {}
for nm, cols in [("champ99", SET_CHAMP), ("lean", SET_LEAN), ("core10", SET_CORE)]:
    st = oof_xgb(cols, XGB_ROUNDS, STABLE, df, y)
    sd = [oof_xgb(cols, XGB_ROUNDS, SEED_SP[s], df, y) for s in SEEDS]
    lotcv[nm] = dict(stable=round(st, 3), seed_mean=round(float(np.mean(sd)), 3),
                     seed_worst=round(float(np.max(sd)), 3), R2=r2_honest(st), n=len(cols))
    print(f"  {nm:<8} n={len(cols):>3} stable={st:.3f}  seed_mean={np.mean(sd):.3f}  worst={np.max(sd):.3f}  R2={r2_honest(st)}")


=== lot-CV (stable GKF + seed1/2/3) — 챔피언 파라미터 공통 ===
  champ99  n= 99 stable=66.743  seed_mean=66.978  worst=67.661  R2=0.935
  lean     n= 85 stable=66.830  seed_mean=66.956  worst=67.592  R2=0.9348
  core10   n= 10 stable=67.160  seed_mean=67.451  worst=68.310  R2=0.9341


In [5]:
# [셀5] 학습기 내부 시간견고성 (early→late, 홀드아웃 무소비)
print("=== 시간견고성 (train early-Dec → predict late-Dec, ≤2019-01-14) ===")
timerob = {}
for nm, cols in [("champ99", SET_CHAMP), ("lean", SET_LEAN), ("core10", SET_CORE)]:
    m = make_xgb(XGB_PARAMS, XGB_ROUNDS); m.fit(df[cols].iloc[E], y[E])
    r = _rmse(y[L], m.predict(df[cols].iloc[L]))
    gap = r - lotcv[nm]["stable"]
    timerob[nm] = dict(time_rmse=round(r, 3), vs_lotcv_gap=round(gap, 3))
    print(f"  {nm:<8} 시간 RMSE={r:.3f}  (lot-CV 대비 gap {gap:+.3f})")
best = min(timerob, key=lambda k: timerob[k]["time_rmse"])
print(f"  → 시간견고성 최우: {best} (작을수록 드리프트 강건)")


=== 시간견고성 (train early-Dec → predict late-Dec, ≤2019-01-14) ===
  champ99  시간 RMSE=612.911  (lot-CV 대비 gap +546.168)
  lean     시간 RMSE=612.583  (lot-CV 대비 gap +545.753)
  core10   시간 RMSE=617.687  (lot-CV 대비 gap +550.527)
  → 시간견고성 최우: lean (작을수록 드리프트 강건)


In [6]:
# [셀6] 요약 + 저장
print("=== 요약 (참고 — 판정은 강건이) ===")
print(f"{'셋':<9}{'n':>4}{'lotCV_stable':>14}{'seed_mean':>11}{'시간RMSE':>10}{'시간gap':>9}")
for nm in ["champ99", "lean", "core10"]:
    l = lotcv[nm]; t = timerob[nm]
    print(f"{nm:<9}{l['n']:>4}{l['stable']:>14}{l['seed_mean']:>11}{t['time_rmse']:>10}{t['vs_lotcv_gap']:>9}")
print("\n해석 가이드: lean이 champ99 대비 (a) lot-CV 유지(±0.3내) ∧ (b) 시간RMSE↓ 면 = 드리프트 견고 개선.")

out = dict(sets=dict(champ99=len(SET_CHAMP), lean=len(SET_LEAN), core10=len(SET_CORE)),
           lot_cv=lotcv, time_robustness=timerob,
           note="학습기 내부 시간분할(홀드아웃 무소비). 최종 시간판정=M8 B2′ 롤링-재학습 1회. 판정은 강건이 회신 숫자로.")
json.dump(out, open("lean_eval_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nsaved: lean_eval_results.json")


=== 요약 (참고 — 판정은 강건이) ===
셋           n  lotCV_stable  seed_mean    시간RMSE    시간gap
champ99    99        66.743     66.978   612.911  546.168
lean       85         66.83     66.956   612.583  545.753
core10     10         67.16     67.451   617.687  550.527

해석 가이드: lean이 champ99 대비 (a) lot-CV 유지(±0.3내) ∧ (b) 시간RMSE↓ 면 = 드리프트 견고 개선.

saved: lean_eval_results.json
